In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!rm -rf /content/yolo_dataset
!unzip -q /content/drive/MyDrive/yolo_dataset_v3.zip -d /content/
!pip install ultralytics -q

import os
print("train tiles :", len(os.listdir('/content/yolo_dataset/images/train')))
print("val tiles   :", len(os.listdir('/content/yolo_dataset/images/val')))

train tiles : 730
val tiles   : 289


In [ ]:
import yaml

path = '/content/yolo_dataset/data.yaml'
with open(path) as f:
    cfg = yaml.safe_load(f)

# data.yaml was written on Windows and carries a C:\ path that means
# nothing inside Colab. Overwrite it with the Linux location.
cfg['path']  = '/content/yolo_dataset'
cfg['train'] = 'images/train'
cfg['val']   = 'images/val'

with open(path, 'w') as f:
    yaml.dump(cfg, f, sort_keys=False)

print("path :", cfg['path'])
print("nc   :", cfg['nc'])

path : /content/yolo_dataset
nc   : 375


In [ ]:
"""
Clean YOLOv8s baseline on the bug-fixed dataset.

Deliberately NOT carrying over from the previous run:
  - synthetic copy-paste / whole-tile oversampling
  - the weighted train.txt
  - the train/val split rebuild
All of those were workarounds for a dataset that had 92% of its annotations
destroyed by a coordinate-convention bug. With 7,450 real training boxes
instead of 383, we need a clean baseline before deciding what, if anything,
still needs compensating for.

Settings that ARE deliberate:
  fliplr / flipud = 0.0 : cuneiform is directional; a mirrored sign is either
                          a different sign or no sign at all, so flipping
                          would teach the model wrong shape-label pairings.
  imgsz = 512           : matches the tile size exactly, no resampling.
  AdamW, lr0 = 0.001    : the auto-selected 2.8e-05 in an earlier run was far
                          too low to train a freshly initialised 375-class head.
The train/val split is at TABLET level (47 train tablets vs 34 held-out test
tablets), so no tile from a training tablet can appear in validation.
"""
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data='/content/yolo_dataset/data.yaml',
    epochs=150,
    imgsz=512,
    batch=16,
    optimizer='AdamW',
    lr0=0.001,
    fliplr=0.0,
    flipud=0.0,
    patience=40,
    seed=0,
    project='/content/runs',
    name='domain1_bugfix_v2',
    plots=True,
)

print("\nBest weights:", f"{results.save_dir}/weights/best.pt")

Ultralytics 8.4.143 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=domain1_bugfix_v2, nbs=64, nms=None, opset=

In [ ]:
!cp -r /content/runs/domain1_bugfix_v2 /content/drive/MyDrive/domain1_bugfix_v2
print("Saved to MyDrive/domain1_bugfix_v2")

Saved to MyDrive/domain1_bugfix_v2
